In [ ]:
# ── COLAB SETUP ──
import subprocess
import sys

try:
    import torch
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch", "-q"])
    import torch

import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)

# Generate double-well potential dataset
X_dwell = np.linspace(-2, 2, 500)
y_dwell = (X_dwell**2 - 1)**2 + np.random.normal(0, 0.05, 500)

# Generate 2D classification dataset
np.random.seed(42)
X_class_0 = np.random.normal([-1, -0.5], 0.4, (150, 2))
X_class_1 = np.random.normal([1, 0.5], 0.4, (150, 2))
X_class = np.vstack([X_class_0, X_class_1])
y_class = np.hstack([np.zeros(150), np.ones(150)])

print(f"X_dwell shape: {X_dwell.shape}")
print(f"y_dwell shape: {y_dwell.shape}")
print(f"X_class shape: {X_class.shape}")
print(f"y_class shape: {y_class.shape}")

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jolayfield/chem-lab-tutorials/blob/main/Track3_PLUMED/nn_intro_pytorch.ipynb)

# Introduction to Neural Networks with PyTorch

## Table of Contents

1. [The Single Neuron](#section-1)
2. [Activation Functions](#section-2)
3. [The Forward Pass](#section-3)
4. [Loss Functions](#section-4)
5. [Backpropagation](#section-5)
6. [Training with PyTorch](#section-6)
7. [Fitting a Double-Well Potential](#section-7)
8. [Classifying Molecular Conformations](#section-8)
9. [Neural Network Potentials](#section-9)
10. [NNs for Collective Variables](#section-10)
11. [Further Reading](#section-11)

## 1. The Single Neuron

The fundamental building block of a neural network is the **neuron** (or **perceptron**). A single neuron applies a weighted sum of inputs plus a bias term, then passes the result through an activation function.

The pre-activation (sometimes called the "pre-nonlinearity" or "logit") for a single neuron is:

$$z = \sum_{i=1}^{n} w_i x_i + b$$

where:
- $x_i$ are the input features (e.g., molecular coordinates)
- $w_i$ are **weights** that scale the importance of each input
- $b$ is the **bias** term

The output is then:

$$\hat{y} = \sigma(z)$$

where $\sigma$ is an **activation function**.

### Geometric Interpretation

In one dimension, the equation $z = wx + b = 0$ defines the decision threshold at $x = -b/w$. The weight $w$ controls the slope of the transition, while the bias $b$ shifts this threshold. For $w > 0$, increasing $b$ shifts the threshold to the left.

In [ ]:
def neuron_numpy(x, w, b, activation='sigmoid'):
    """
    Compute output of a single neuron.
    
    Args:
        x: input (scalar or array)
        w: weight (scalar or array, same shape as x)
        b: bias (scalar)
        activation: 'sigmoid', 'tanh', 'relu'
    
    Returns:
        output after activation
    """
    z = np.sum(w * x) + b
    
    if activation == 'sigmoid':
        return 1 / (1 + np.exp(-z))
    elif activation == 'tanh':
        return np.tanh(z)
    elif activation == 'relu':
        return np.maximum(0, z)
    else:
        raise ValueError(f"Unknown activation: {activation}")

# Example: w = 2, b = -1 on 1D input
w, b = 2.0, -1.0
x_vals = np.linspace(-4, 4, 200)
y_sigmoid = np.array([neuron_numpy(xi, w, b, 'sigmoid') for xi in x_vals])

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.plot(x_vals, y_sigmoid, 'b-', linewidth=2, label=f'sigmoid(z), w={w}, b={b}')
ax.axvline(-b/w, color='red', linestyle='--', linewidth=1.5, label=f'Decision threshold: x = {-b/w}')
ax.set_xlabel('Input x', fontsize=12)
ax.set_ylabel('Output', fontsize=12)
ax.set_title('Single Neuron with Sigmoid Activation', fontsize=13)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f"Threshold at x = {-b/w:.2f}")
print(f"Output at threshold: {neuron_numpy(-b/w, w, b, 'sigmoid'):.4f}")

In [ ]:
# PyTorch equivalent: nn.Linear(1, 1) + nn.Sigmoid()
class SingleNeuron(nn.Module):
    def __init__(self, w_init, b_init):
        super().__init__()
        self.linear = nn.Linear(1, 1, bias=True)
        self.sigmoid = nn.Sigmoid()
        # Set weights manually
        with torch.no_grad():
            self.linear.weight.copy_(torch.tensor([[w_init]], dtype=torch.float32))
            self.linear.bias.copy_(torch.tensor([b_init], dtype=torch.float32))
    
    def forward(self, x):
        return self.sigmoid(self.linear(x))

# Create model with same weights as numpy example
model = SingleNeuron(2.0, -1.0)

# Compare outputs
x_torch = torch.from_numpy(x_vals.reshape(-1, 1)).float()
with torch.no_grad():
    y_torch = model(x_torch).numpy().flatten()

# Plot both
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.plot(x_vals, y_sigmoid, 'b-', linewidth=2.5, label='NumPy', alpha=0.8)
ax.plot(x_vals, y_torch, 'r--', linewidth=2, label='PyTorch', alpha=0.7)
ax.set_xlabel('Input x', fontsize=12)
ax.set_ylabel('Output', fontsize=12)
ax.set_title('Single Neuron: NumPy vs PyTorch', fontsize=13)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

# Verify they match
max_diff = np.max(np.abs(y_sigmoid - y_torch))
print(f"Max difference between NumPy and PyTorch: {max_diff:.2e}")

## 2. Activation Functions

Activation functions introduce nonlinearity, allowing neural networks to learn complex mappings. Without them, stacking layers would be equivalent to a single linear transformation. Below are the most common activations used in chemistry applications.

### Sigmoid
$$\sigma(x) = \frac{1}{1 + e^{-x}}$$
$$\frac{d\sigma}{dx} = \sigma(x)(1 - \sigma(x))$$

### Hyperbolic Tangent (tanh)
$$\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$$
$$\frac{d\tanh}{dx} = 1 - \tanh^2(x)$$

### Rectified Linear Unit (ReLU)
$$\text{ReLU}(x) = \max(0, x)$$
$$\frac{d\text{ReLU}}{dx} = \begin{cases} 1 & \text{if } x > 0 \\ 0 & \text{if } x \leq 0 \end{cases}$$

### Softplus
$$\text{Softplus}(x) = \log(1 + e^x)$$
$$\frac{d\text{Softplus}}{dx} = \frac{1}{1 + e^{-x}} = \sigma(x)$$

### Gaussian Error Linear Unit (GELU)
$$\text{GELU}(x) \approx 0.5 x \left(1 + \tanh\left(\sqrt{\frac{2}{\pi}}(x + 0.044715 x^3)\right)\right)$$
$$\frac{d\text{GELU}}{dx} \approx 0.5 \left(1 + \tanh(\cdots)\right) + 0.5 x \cdot \text{sech}^2(\cdots) \cdot \text{poly}(x)$$

### Leaky ReLU
$$\text{LeakyReLU}(x) = \begin{cases} x & \text{if } x > 0 \\ 0.01 x & \text{if } x \leq 0 \end{cases}$$
$$\frac{d\text{LeakyReLU}}{dx} = \begin{cases} 1 & \text{if } x > 0 \\ 0.01 & \text{if } x \leq 0 \end{cases}$$

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_grad(x):
    s = sigmoid(x)
    return s * (1 - s)

def tanh_grad(x):
    return 1 - np.tanh(x)**2

def relu(x):
    return np.maximum(0, x)

def relu_grad(x):
    return (x > 0).astype(float)

def softplus(x):
    return np.log(1 + np.exp(x))

def softplus_grad(x):
    return sigmoid(x)

def gelu_approx(x):
    cdf = 0.5 * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))
    return x * cdf

def gelu_approx_grad(x):
    a = np.sqrt(2/np.pi) * (x + 0.044715 * x**3)
    cdf = 0.5 * (1 + np.tanh(a))
    pdf_term = np.sqrt(2/np.pi) * (1 + 3 * 0.044715 * x**2)
    dpdf = 0.5 * (1 - np.tanh(a)**2) * pdf_term
    return cdf + x * dpdf

def leaky_relu(x, alpha=0.01):
    return np.where(x > 0, x, alpha * x)

def leaky_relu_grad(x, alpha=0.01):
    return np.where(x > 0, 1, alpha)

x = np.linspace(-4, 4, 200)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

activations = [
    ('Sigmoid', sigmoid, sigmoid_grad),
    ('Tanh', np.tanh, tanh_grad),
    ('ReLU', relu, relu_grad),
    ('Softplus', softplus, softplus_grad),
    ('GELU (approx)', gelu_approx, gelu_approx_grad),
    ('Leaky ReLU', leaky_relu, leaky_relu_grad)
]

for idx, (name, func, grad_func) in enumerate(activations):
    row, col = idx // 3, idx % 3
    ax = axes[row, col]
    
    y = func(x)
    dy = grad_func(x)
    
    ax2 = ax.twinx()
    line1 = ax.plot(x, y, 'b-', linewidth=2.5, label=f'{name}(x)')
    line2 = ax2.plot(x, dy, 'r--', linewidth=2, label=f"{name}'(x)")
    
    ax.set_xlabel('x', fontsize=10)
    ax.set_ylabel('f(x)', color='b', fontsize=10)
    ax2.set_ylabel("f'(x)", color='r', fontsize=10)
    ax.tick_params(axis='y', labelcolor='b')
    ax2.tick_params(axis='y', labelcolor='r')
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_ylim([y.min() - 0.2, y.max() + 0.2])

plt.tight_layout()
plt.show()

In [ ]:
# Verify gradients using torch.autograd
x_test = 1.5
x_torch = torch.tensor([x_test], dtype=torch.float32, requires_grad=True)

results = []

# Sigmoid
y_sig = torch.sigmoid(x_torch)
y_sig.backward()
grad_sig_torch = x_torch.grad.item()
grad_sig_analytical = sigmoid_grad(x_test)
results.append(('Sigmoid', grad_sig_analytical, grad_sig_torch))

# Tanh
x_torch.grad = None
y_tanh = torch.tanh(x_torch)
y_tanh.backward()
grad_tanh_torch = x_torch.grad.item()
grad_tanh_analytical = tanh_grad(x_test)
results.append(('Tanh', grad_tanh_analytical, grad_tanh_torch))

# ReLU
x_torch.grad = None
y_relu = torch.relu(x_torch)
y_relu.backward()
grad_relu_torch = x_torch.grad.item()
grad_relu_analytical = relu_grad(x_test)
results.append(('ReLU', grad_relu_analytical, grad_relu_torch))

# Softplus
x_torch.grad = None
y_softplus = torch.nn.functional.softplus(x_torch)
y_softplus.backward()
grad_softplus_torch = x_torch.grad.item()
grad_softplus_analytical = softplus_grad(x_test)
results.append(('Softplus', grad_softplus_analytical, grad_softplus_torch))

# GELU
x_torch.grad = None
y_gelu = torch.nn.functional.gelu(x_torch)
y_gelu.backward()
grad_gelu_torch = x_torch.grad.item()
grad_gelu_analytical = gelu_approx_grad(x_test)
results.append(('GELU', grad_gelu_analytical, grad_gelu_torch))

# Leaky ReLU
x_torch.grad = None
y_lrelu = torch.nn.functional.leaky_relu(x_torch, negative_slope=0.01)
y_lrelu.backward()
grad_lrelu_torch = x_torch.grad.item()
grad_lrelu_analytical = leaky_relu_grad(x_test, alpha=0.01)
results.append(('Leaky ReLU', grad_lrelu_analytical, grad_lrelu_torch))

print(f"Gradient verification at x = {x_test}\n")
print(f"{'Activation':<15} {'Analytical':<20} {'PyTorch':<20} {'Diff':<15}")
print("-" * 70)
for name, analytical, pytorch in results:
    diff = abs(analytical - pytorch)
    print(f"{name:<15} {analytical:<20.10f} {pytorch:<20.10f} {diff:<15.2e}")

## 3. The Forward Pass

A multi-layer neural network transforms input through a sequence of layers. Each layer applies a linear transformation followed by a nonlinear activation.

For a 2-layer network:

**Layer 1 (hidden layer):**
$$\mathbf{z}^{(1)} = \mathbf{W}^{(1)} \mathbf{x} + \mathbf{b}^{(1)}$$
$$\mathbf{h} = \sigma(\mathbf{z}^{(1)})$$

**Layer 2 (output layer):**
$$\mathbf{z}^{(2)} = \mathbf{W}^{(2)} \mathbf{h} + \mathbf{b}^{(2)}$$
$$\hat{\mathbf{y}} = \sigma(\mathbf{z}^{(2)})$$

### Numerical Example

Let $\sigma = \tanh$ and:
- Input: $\mathbf{x} = [1.0, 0.5, -1.0]^T$
- Layer 1: $\mathbf{W}^{(1)} = \begin{bmatrix} 0.5 & -0.3 & 0.8 \\ 0.2 & 0.6 & -0.4 \end{bmatrix}$ (shape 2x3), $\mathbf{b}^{(1)} = [0.1, -0.2]^T$
- Layer 2: $\mathbf{W}^{(2)} = [0.7, -0.5]$ (shape 1x2), $\mathbf{b}^{(2)} = [0.3]$

**Compute $\mathbf{z}^{(1)}$:**
$$z_1^{(1)} = 0.5(1.0) + (-0.3)(0.5) + 0.8(-1.0) + 0.1 = 0.5 - 0.15 - 0.8 + 0.1 = -0.35$$
$$z_2^{(1)} = 0.2(1.0) + 0.6(0.5) + (-0.4)(-1.0) + (-0.2) = 0.2 + 0.3 + 0.4 - 0.2 = 0.7$$

**Compute $\mathbf{h} = \tanh(\mathbf{z}^{(1)})$:**
$$h_1 = \tanh(-0.35) \approx -0.3365$$
$$h_2 = \tanh(0.7) \approx 0.6042$$

**Compute $\mathbf{z}^{(2)}$:**
$$z_1^{(2)} = 0.7(-0.3365) + (-0.5)(0.6042) + 0.3 = -0.2356 - 0.3021 + 0.3 = -0.2377$$

**Compute $\hat{y} = \tanh(\mathbf{z}^{(2)})$:**
$$\hat{y} = \tanh(-0.2377) \approx -0.2347$$

In [ ]:
# Numerical example: forward pass with manual computation
x = np.array([1.0, 0.5, -1.0])
W1 = np.array([[0.5, -0.3, 0.8],
                [0.2, 0.6, -0.4]])
b1 = np.array([0.1, -0.2])
W2 = np.array([[0.7, -0.5]])
b2 = np.array([0.3])

# Forward pass
z1 = W1 @ x + b1
h = np.tanh(z1)
z2 = W2 @ h + b2
y_hat = np.tanh(z2)

print("Numerical Forward Pass Example")
print("=" * 50)
print(f"Input x: {x}")
print(f"\nLayer 1 weights W1 shape: {W1.shape}")
print(f"Layer 1 bias b1: {b1}")
print(f"\nz1 = W1 @ x + b1: {z1}")
print(f"h = tanh(z1): {h}")
print(f"\nLayer 2 weights W2 shape: {W2.shape}")
print(f"Layer 2 bias b2: {b2}")
print(f"\nz2 = W2 @ h + b2: {z2}")
print(f"y_hat = tanh(z2): {y_hat}")

In [ ]:
# PyTorch equivalent with manually set weights
class TwoLayerNet(nn.Module):
    def __init__(self, W1, b1, W2, b2):
        super().__init__()
        self.fc1 = nn.Linear(3, 2, bias=True)
        self.fc2 = nn.Linear(2, 1, bias=True)
        self.activation = nn.Tanh()
        
        # Set weights manually
        with torch.no_grad():
            self.fc1.weight.copy_(torch.from_numpy(W1).float())
            self.fc1.bias.copy_(torch.from_numpy(b1).float())
            self.fc2.weight.copy_(torch.from_numpy(W2).float())
            self.fc2.bias.copy_(torch.from_numpy(b2).float())
    
    def forward(self, x):
        z1 = self.fc1(x)
        h = self.activation(z1)
        z2 = self.fc2(h)
        return self.activation(z2), z1, h

model = TwoLayerNet(W1, b1, W2, b2)

# Forward pass
x_torch = torch.from_numpy(x).float().unsqueeze(0)  # Shape (1, 3)
with torch.no_grad():
    y_hat_torch, z1_torch, h_torch = model(x_torch)

# Compare
print("\nPyTorch Verification")
print("=" * 50)
print(f"z1 (NumPy):   {z1}")
print(f"z1 (PyTorch): {z1_torch.squeeze().numpy()}")
print(f"z1 match: {np.allclose(z1, z1_torch.squeeze().numpy())}")
print(f"\nh (NumPy):   {h}")
print(f"h (PyTorch): {h_torch.squeeze().numpy()}")
print(f"h match: {np.allclose(h, h_torch.squeeze().numpy())}")
print(f"\nz2 (NumPy):   {z2}")
print(f"z2 (PyTorch): (computed internally)")
print(f"\ny_hat (NumPy):   {y_hat}")
print(f"y_hat (PyTorch): {y_hat_torch.squeeze().numpy()}")
print(f"y_hat match: {np.allclose(y_hat, y_hat_torch.squeeze().numpy())}")

## 4. Loss Functions

The loss function quantifies the discrepancy between predictions and targets. The network learns by minimizing this loss.

### Mean Squared Error (MSE)

Used for regression problems:
$$L_{\text{MSE}} = \frac{1}{2}(\hat{y} - y)^2$$

Gradient w.r.t. prediction:
$$\frac{\partial L}{\partial \hat{y}} = \hat{y} - y$$

### Binary Cross-Entropy (BCE)

Used for binary classification. If $\hat{y}$ is the predicted probability (post-sigmoid):
$$L_{\text{BCE}} = -[y \log(\hat{y}) + (1-y) \log(1-\hat{y})]$$

Gradient w.r.t. prediction:
$$\frac{\partial L}{\partial \hat{y}} = \frac{\hat{y} - y}{\hat{y}(1-\hat{y})}$$

Note: PyTorch's `BCEWithLogitsLoss` applies sigmoid internally and uses a numerically stable formula, combining the sigmoid and BCE loss to avoid computing log of very small numbers.

In [ ]:
def mse_loss(y_hat, y):
    """
    Mean squared error loss: L = 0.5 * (y_hat - y)^2
    """
    return 0.5 * (y_hat - y)**2

def mse_grad(y_hat, y):
    """
    Gradient of MSE w.r.t. y_hat
    """
    return y_hat - y

def bce_loss(y_hat, y):
    """
    Binary cross-entropy loss (assuming y_hat is already post-sigmoid)
    """
    eps = 1e-7
    y_hat = np.clip(y_hat, eps, 1 - eps)
    return -(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))

def bce_grad(y_hat, y):
    """
    Gradient of BCE w.r.t. y_hat
    """
    eps = 1e-7
    y_hat = np.clip(y_hat, eps, 1 - eps)
    return (y_hat - y) / (y_hat * (1 - y_hat))

# Plot loss and gradient as functions of y_hat, with y=1.0
y = 1.0
y_hat = np.linspace(0.01, 0.99, 200)

mse = mse_loss(y_hat, y)
mse_g = mse_grad(y_hat, y)
bce = bce_loss(y_hat, y)
bce_g = bce_grad(y_hat, y)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MSE
ax = axes[0]
ax2 = ax.twinx()
line1 = ax.plot(y_hat, mse, 'b-', linewidth=2.5, label='Loss')
line2 = ax2.plot(y_hat, mse_g, 'r--', linewidth=2, label='Gradient')
ax.set_xlabel('Prediction ($\\hat{y}$)', fontsize=11)
ax.set_ylabel('MSE Loss', color='b', fontsize=11)
ax2.set_ylabel('Gradient', color='r', fontsize=11)
ax.tick_params(axis='y', labelcolor='b')
ax2.tick_params(axis='y', labelcolor='r')
ax.set_title(f'MSE Loss and Gradient (y={y})', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.axvline(y, color='gray', linestyle=':', linewidth=1.5)

# BCE
ax = axes[1]
ax2 = ax.twinx()
line1 = ax.plot(y_hat, bce, 'b-', linewidth=2.5, label='Loss')
line2 = ax2.plot(y_hat, bce_g, 'r--', linewidth=2, label='Gradient')
ax.set_xlabel('Prediction ($\\hat{y}$)', fontsize=11)
ax.set_ylabel('BCE Loss', color='b', fontsize=11)
ax2.set_ylabel('Gradient', color='r', fontsize=11)
ax.tick_params(axis='y', labelcolor='b')
ax2.tick_params(axis='y', labelcolor='r')
ax.set_title(f'BCE Loss and Gradient (y={y})', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.axvline(y, color='gray', linestyle=':', linewidth=1.5)
ax.set_ylim([0, 5])

plt.tight_layout()
plt.show()

## 5. Backpropagation

Backpropagation computes gradients of the loss with respect to all parameters using the chain rule. This is the core algorithm that enables training.

### Chain Rule Review

For a composition of functions $L(f(g(x)))$:
$$\frac{dL}{dx} = \frac{dL}{df} \cdot \frac{df}{dg} \cdot \frac{dg}{dx}$$

### Full Derivation for 2-Layer Network with MSE Loss

Using the same network from Section 3 with MSE loss and target $y = 0.5$:

**Forward pass (already computed):**
- $\mathbf{z}^{(1)} = [-0.35, 0.7]^T$
- $\mathbf{h} = [-0.3365, 0.6042]^T$
- $\mathbf{z}^{(2)} = [-0.2377]$
- $\hat{y} = -0.2347$

**Loss:**
$$L = \frac{1}{2}(\hat{y} - y)^2 = \frac{1}{2}(-0.2347 - 0.5)^2 = \frac{1}{2}(-0.7347)^2 = 0.2699$$

**Backprop through output layer:**

The output error is:
$$\delta^{\text{out}} = \frac{\partial L}{\partial \hat{y}} = \hat{y} - y = -0.2347 - 0.5 = -0.7347$$

But we also need the gradient w.r.t. $z^{(2)}$ (before activation). Since $\hat{y} = \tanh(z^{(2)})$:
$$\delta^{(2)} = \frac{\partial L}{\partial z^{(2)}} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z^{(2)}} = \delta^{\text{out}} \cdot (1 - \hat{y}^2)$$
$$\delta^{(2)} = -0.7347 \cdot (1 - (-0.2347)^2) = -0.7347 \cdot (1 - 0.0551) = -0.7347 \cdot 0.9449 = -0.6940$$

**Gradients w.r.t. Layer 2 parameters:**

$$\frac{\partial L}{\partial \mathbf{W}^{(2)}} = \delta^{(2)} \cdot \mathbf{h}^T = -0.6940 \cdot [-0.3365, 0.6042]^T$$
$$= [(-0.6940) \times (-0.3365), (-0.6940) \times 0.6042]^T = [0.2335, -0.4191]$$

$$\frac{\partial L}{\partial \mathbf{b}^{(2)}} = \delta^{(2)} = -0.6940$$

**Backprop to hidden layer:**

$$\delta^{(1)} = \frac{\partial L}{\partial \mathbf{h}} = (\mathbf{W}^{(2)})^T \cdot \delta^{(2)} = [0.7, -0.5]^T \cdot (-0.6940)$$
$$= [0.7 \times (-0.6940), -0.5 \times (-0.6940)]^T = [-0.4858, 0.3470]^T$$

Now apply the activation derivative:
$$\frac{\partial L}{\partial \mathbf{z}^{(1)}} = \delta^{(1)} \odot (1 - \mathbf{h}^2)$$

where $\odot$ denotes element-wise multiplication.

$(1 - h_1^2) = 1 - (-0.3365)^2 = 1 - 0.1133 = 0.8867$

$(1 - h_2^2) = 1 - (0.6042)^2 = 1 - 0.3651 = 0.6349$

$$\frac{\partial L}{\partial \mathbf{z}^{(1)}} = [-0.4858 \times 0.8867, 0.3470 \times 0.6349]^T = [-0.4307, 0.2203]^T$$

**Gradients w.r.t. Layer 1 parameters:**

$$\frac{\partial L}{\partial \mathbf{W}^{(1)}} = \frac{\partial L}{\partial \mathbf{z}^{(1)}} \cdot \mathbf{x}^T$$

This is a 2x1 vector times a 1x3 vector (outer product):
$$\frac{\partial L}{\partial \mathbf{W}^{(1)}} = \begin{bmatrix} -0.4307 \\ 0.2203 \end{bmatrix} \begin{bmatrix} 1.0 & 0.5 & -1.0 \end{bmatrix}$$

$$= \begin{bmatrix} -0.4307 & -0.2154 & 0.4307 \\ 0.2203 & 0.1102 & -0.2203 \end{bmatrix}$$

$$\frac{\partial L}{\partial \mathbf{b}^{(1)}} = [-0.4307, 0.2203]^T$$

In [ ]:
# Manual backprop with exact numerics from Section 3
# Forward values (from previous example)
x = np.array([1.0, 0.5, -1.0])
W1 = np.array([[0.5, -0.3, 0.8],
                [0.2, 0.6, -0.4]])
b1 = np.array([0.1, -0.2])
W2 = np.array([[0.7, -0.5]])
b2 = np.array([0.3])

# Forward pass
z1 = W1 @ x + b1
h = np.tanh(z1)
z2 = W2 @ h + b2
y_hat = np.tanh(z2)
y = 0.5

# Loss
L = 0.5 * (y_hat - y)**2

# Backward pass
# dL/dy_hat
dL_dyhat = y_hat - y  # Scalar

# dL/dz2 = dL/dyhat * dyhat/dz2
dyhat_dz2 = 1 - y_hat**2
dL_dz2 = dL_dyhat * dyhat_dz2  # Scalar

# dL/dW2 = dL/dz2 * dz2/dW2 = dL/dz2 * h^T
dL_dW2 = dL_dz2 * h.reshape(1, -1)  # Shape (1, 2)

# dL/db2 = dL/dz2
dL_db2 = dL_dz2  # Scalar

# dL/dh = W2^T * dL/dz2
dL_dh = W2.T * dL_dz2  # Shape (2,)

# dL/dz1 = dL/dh * dh/dz1 (element-wise)
dh_dz1 = 1 - h**2
dL_dz1 = dL_dh * dh_dz1  # Shape (2,)

# dL/dW1 = dL/dz1 * x^T
dL_dW1 = dL_dz1.reshape(-1, 1) @ x.reshape(1, -1)  # Shape (2, 3)

# dL/db1 = dL/dz1
dL_db1 = dL_dz1  # Shape (2,)

print("Manual Backprop Computation")
print("=" * 60)
print(f"Target y: {y}")
print(f"Prediction y_hat: {y_hat:.6f}")
print(f"Loss L: {L:.6f}")
print(f"\nBackprop values:")
print(f"dL/dyhat: {dL_dyhat:.6f}")
print(f"dL/dz2: {dL_dz2:.6f}")
print(f"dL/dW2: {dL_dW2}")
print(f"dL/db2: {dL_db2:.6f}")
print(f"dL/dh: {dL_dh}")
print(f"dL/dz1: {dL_dz1}")
print(f"dL/dW1:\n{dL_dW1}")
print(f"dL/db1: {dL_db1}")

In [ ]:
# PyTorch autograd verification
x_pt = torch.from_numpy(x).float().unsqueeze(0)  # Shape (1, 3)
y_pt = torch.tensor([y], dtype=torch.float32)

# Create model with same weights
model = TwoLayerNet(W1, b1, W2, b2)

# Enable gradients
for param in model.parameters():
    param.requires_grad = True

# Forward pass
y_hat_pt, z1_pt, h_pt = model(x_pt)

# Loss
loss = 0.5 * (y_hat_pt - y_pt)**2
loss.backward()

# Extract gradients
grad_W2_pt = model.fc2.weight.grad.numpy()
grad_b2_pt = model.fc2.bias.grad.numpy()
grad_W1_pt = model.fc1.weight.grad.numpy()
grad_b1_pt = model.fc1.bias.grad.numpy()

# Compare
print("\nComparison: Manual vs PyTorch Autograd")
print("=" * 60)

print(f"\ndL/dW2:")
print(f"  Manual:   {dL_dW2}")
print(f"  PyTorch:  {grad_W2_pt}")
print(f"  Match: {np.allclose(dL_dW2, grad_W2_pt)}")

print(f"\ndL/db2:")
print(f"  Manual:   {dL_db2}")
print(f"  PyTorch:  {grad_b2_pt}")
print(f"  Match: {np.allclose(dL_db2, grad_b2_pt)}")

print(f"\ndL/dW1:")
print(f"  Manual:\n{dL_dW1}")
print(f"  PyTorch:\n{grad_W1_pt}")
print(f"  Match: {np.allclose(dL_dW1, grad_W1_pt)}")

print(f"\ndL/db1:")
print(f"  Manual:   {dL_db1}")
print(f"  PyTorch:  {grad_b1_pt}")
print(f"  Match: {np.allclose(dL_db1, grad_b1_pt)}")

print(f"\nAll gradients match: {np.allclose(dL_dW2, grad_W2_pt) and np.allclose(dL_db2, grad_b2_pt) and np.allclose(dL_dW1, grad_W1_pt) and np.allclose(dL_db1, grad_b1_pt)}")

## 6. Training with PyTorch

Training iteratively computes gradients and updates parameters to minimize loss. The basic update rule is:

$$\mathbf{w} \leftarrow \mathbf{w} - \eta \frac{\partial L}{\partial \mathbf{w}}$$

where $\eta$ is the learning rate.

### Adam Optimizer

Adam (Adaptive Moment Estimation) maintains two running statistics:

$$\mathbf{m}_t = \beta_1 \mathbf{m}_{t-1} + (1 - \beta_1) \frac{\partial L}{\partial \mathbf{w}}$$

$$\mathbf{v}_t = \beta_2 \mathbf{v}_{t-1} + (1 - \beta_2) \left(\frac{\partial L}{\partial \mathbf{w}}\right)^2$$

With bias correction:

$$\hat{\mathbf{m}}_t = \frac{\mathbf{m}_t}{1 - \beta_1^t}, \quad \hat{\mathbf{v}}_t = \frac{\mathbf{v}_t}{1 - \beta_2^t}$$

Update:

$$\mathbf{w} \leftarrow \mathbf{w} - \eta \frac{\hat{\mathbf{m}}_t}{\sqrt{\hat{\mathbf{v}}_t} + \epsilon}$$

Typical values: $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\epsilon = 10^{-8}$.

### Training Loop Structure

1. Compute forward pass on mini-batch
2. Compute loss
3. Zero gradients
4. Backward pass
5. Update parameters via optimizer
6. Track metrics

In [ ]:
def train_model(model, X_train, y_train, X_val, y_val, n_epochs, lr, loss_fn, device='cpu'):
    """
    Train a neural network model.
    
    Args:
        model: PyTorch nn.Module
        X_train: training features, numpy array or torch tensor
        y_train: training targets
        X_val: validation features
        y_val: validation targets
        n_epochs: number of training epochs
        lr: learning rate
        loss_fn: loss function (e.g., nn.MSELoss())
        device: 'cpu' or 'cuda'
    
    Returns:
        train_losses: list of training losses per epoch
        val_losses: list of validation losses per epoch
    """
    # Convert to tensors if needed
    if isinstance(X_train, np.ndarray):
        X_train = torch.from_numpy(X_train).float()
    if isinstance(y_train, np.ndarray):
        y_train = torch.from_numpy(y_train).float()
    if isinstance(X_val, np.ndarray):
        X_val = torch.from_numpy(X_val).float()
    if isinstance(y_val, np.ndarray):
        y_val = torch.from_numpy(y_val).float()
    
    # Ensure y is 2D if necessary
    if y_train.dim() == 1:
        y_train = y_train.unsqueeze(1)
    if y_val.dim() == 1:
        y_val = y_val.unsqueeze(1)
    
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    train_losses = []
    val_losses = []
    
    X_train = X_train.to(device)
    y_train = y_train.to(device)
    X_val = X_val.to(device)
    y_val = y_val.to(device)
    
    for epoch in range(n_epochs):
        # Training
        model.train()
        optimizer.zero_grad()
        
        y_pred = model(X_train)
        loss_train = loss_fn(y_pred, y_train)
        loss_train.backward()
        optimizer.step()
        
        train_losses.append(loss_train.item())
        
        # Validation
        model.eval()
        with torch.no_grad():
            y_val_pred = model(X_val)
            loss_val = loss_fn(y_val_pred, y_val)
        
        val_losses.append(loss_val.item())
        
        if (epoch + 1) % (n_epochs // 5) == 0:
            print(f"Epoch {epoch + 1}/{n_epochs}: train_loss={train_losses[-1]:.6f}, val_loss={val_losses[-1]:.6f}")
    
    return train_losses, val_losses

print("Train function defined.")

## 7. Fitting a Double-Well Potential

The double-well potential is a classic model system in chemical physics:

$$V(x) = (x^2 - 1)^2$$

This function has two minima at $x = \pm 1$ and a barrier (maximum) at $x = 0$. It appears in molecular systems ranging from models of isomerization to descriptions of phi/psi dihedral angles in proteins. We will train neural networks of varying capacity to fit this potential.

In [ ]:
# Prepare data
n_train = int(0.8 * len(X_dwell))
idx_train = np.arange(n_train)
idx_val = np.arange(n_train, len(X_dwell))

X_dwell_train = X_dwell[idx_train].reshape(-1, 1)
y_dwell_train = y_dwell[idx_train].reshape(-1, 1)
X_dwell_val = X_dwell[idx_val].reshape(-1, 1)
y_dwell_val = y_dwell[idx_val].reshape(-1, 1)

# Define models of increasing complexity
class SmallNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(1, 4)
        self.fc2 = nn.Linear(4, 1)
        self.act = nn.Tanh()
    
    def forward(self, x):
        x = self.act(self.fc1(x))
        return self.fc2(x)

class MediumNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(1, 16)
        self.fc2 = nn.Linear(16, 16)
        self.fc3 = nn.Linear(16, 1)
        self.act = nn.Tanh()
    
    def forward(self, x):
        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        return self.fc3(x)

class LargeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(1, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, 64)
        self.fc4 = nn.Linear(64, 1)
        self.act = nn.Tanh()
    
    def forward(self, x):
        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        x = self.act(self.fc3(x))
        return self.fc4(x)

# Train all three
print("Training double-well potential models...\n")
loss_fn = nn.MSELoss()

models = {
    'Small (4 hidden)': SmallNet(),
    'Medium (16x2)': MediumNet(),
    'Large (64x3)': LargeNet()
}

results = {}
for name, model in models.items():
    print(f"Training {name}...")
    train_loss, val_loss = train_model(model, X_dwell_train, y_dwell_train, 
                                         X_dwell_val, y_dwell_val, n_epochs=1000, lr=0.01, loss_fn=loss_fn)
    results[name] = {'model': model, 'train_loss': train_loss, 'val_loss': val_loss}
    print(f"  Final train loss: {train_loss[-1]:.6f}, val loss: {val_loss[-1]:.6f}\n")

In [ ]:
# Plot results
x_plot = np.linspace(-2.5, 2.5, 300).reshape(-1, 1)
y_true_plot = (x_plot.flatten()**2 - 1)**2

fig, axes = plt.subplots(2, 1, figsize=(12, 10))

# Plot 1: Predictions vs True
ax = axes[0]
colors = ['blue', 'green', 'red']
ax.plot(x_plot, y_true_plot, 'k-', linewidth=3, label='True V(x)', alpha=0.8)
ax.scatter(X_dwell, y_dwell, alpha=0.3, s=20, color='gray', label='Data')

for (name, res), color in zip(results.items(), colors):
    model = res['model']
    model.eval()
    with torch.no_grad():
        x_plot_torch = torch.from_numpy(x_plot).float()
        y_pred = model(x_plot_torch).numpy().flatten()
    ax.plot(x_plot, y_pred, linewidth=2.5, color=color, label=name, alpha=0.8)

ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('V(x)', fontsize=12)
ax.set_title('Double-Well Potential: Model Predictions', fontsize=13, fontweight='bold')
ax.legend(fontsize=11, loc='upper center')
ax.grid(True, alpha=0.3)
ax.set_xlim([-2.5, 2.5])

# Plot 2: Training curves
ax = axes[1]
for (name, res), color in zip(results.items(), colors):
    ax.semilogy(res['train_loss'], linewidth=2.5, color=color, alpha=0.7, label=f"{name} (train)")
    ax.semilogy(res['val_loss'], linewidth=2.5, color=color, alpha=0.7, linestyle='--', label=f"{name} (val)")

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss (log scale)', fontsize=12)
ax.set_title('Training Curves', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='best')
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

# Print final losses
print("\nFinal Performance on Double-Well Potential:")
print("-" * 60)
for name, res in results.items():
    print(f"{name:20s}: train loss = {res['train_loss'][-1]:.6f}, val loss = {res['val_loss'][-1]:.6f}")

## 8. Classifying Molecular Conformations

Many chemical systems exist in discrete conformational states. For example, protein backbone dihedrals (phi, psi) cluster near characteristic values: alpha-helix around (-60°, -45°), beta-sheet around (-135°, +140°). A binary classifier can distinguish these states from unlabeled molecular dynamics trajectories.

In this exercise, we use a 2D Gaussian mixture to simulate this problem: class 0 centered at (-1, -0.5) and class 1 centered at (1, 0.5).

In [ ]:
# Prepare classification data
n_class_train = int(0.8 * len(X_class))
idx_class = np.arange(len(X_class))
np.random.shuffle(idx_class)

X_class_train = X_class[idx_class[:n_class_train]]
y_class_train = y_class[idx_class[:n_class_train]]
X_class_val = X_class[idx_class[n_class_train:]]
y_class_val = y_class[idx_class[n_class_train:]]

# Define classifier
class BinaryClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 16)
        self.fc2 = nn.Linear(16, 16)
        self.fc3 = nn.Linear(16, 1)
        self.act = nn.ReLU()
    
    def forward(self, x):
        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        return self.fc3(x)  # Logit (for BCEWithLogitsLoss)

classifier = BinaryClassifier()

# Train with BCEWithLogitsLoss
print("Training binary classifier...\n")
loss_fn_bce = nn.BCEWithLogitsLoss()
train_loss_list, val_loss_list = train_model(classifier, X_class_train, y_class_train,
                                               X_class_val, y_class_val, n_epochs=500, lr=0.01,
                                               loss_fn=loss_fn_bce)
print()

# Compute accuracy
classifier.eval()
with torch.no_grad():
    X_val_torch = torch.from_numpy(X_class_val).float()
    logits = classifier(X_val_torch).numpy().flatten()
    probs = 1 / (1 + np.exp(-logits))  # Sigmoid
    predictions = (probs > 0.5).astype(int)
    accuracy = np.mean(predictions == y_class_val)

print(f"Validation accuracy: {accuracy:.4f}")

In [ ]:
# Plot decision boundary
xx, yy = np.meshgrid(np.linspace(-2.5, 2.5, 150), np.linspace(-2.5, 2.5, 150))
grid_points = np.column_stack([xx.flatten(), yy.flatten()])
grid_points_torch = torch.from_numpy(grid_points).float()

with torch.no_grad():
    logits_grid = classifier(grid_points_torch).numpy()
    probs_grid = 1 / (1 + np.exp(-logits_grid))

probs_grid = probs_grid.reshape(xx.shape)

fig, ax = plt.subplots(1, 1, figsize=(10, 9))

# Contour plot
contour = ax.contourf(xx, yy, probs_grid, levels=20, cmap='RdBu_r', alpha=0.8)
contour_lines = ax.contour(xx, yy, probs_grid, levels=[0.5], colors='black', linewidths=2)
plt.colorbar(contour, ax=ax, label='Predicted Probability (Class 1)')

# Data points
ax.scatter(X_class[y_class == 0, 0], X_class[y_class == 0, 1], 
          c='blue', s=50, alpha=0.6, edgecolors='darkblue', linewidth=0.5, label='Class 0')
ax.scatter(X_class[y_class == 1, 0], X_class[y_class == 1, 1],
          c='orange', s=50, alpha=0.6, edgecolors='darkorange', linewidth=0.5, label='Class 1')

ax.set_xlabel('Feature 1', fontsize=12)
ax.set_ylabel('Feature 2', fontsize=12)
ax.set_title(f'Binary Classification with Neural Network Decision Boundary\n(Validation Accuracy: {accuracy:.4f})', 
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.2)
ax.set_xlim([-2.5, 2.5])
ax.set_ylim([-2.5, 2.5])

plt.tight_layout()
plt.show()

## 9. Neural Network Potentials

### The Force Field Problem

Classical molecular dynamics uses fixed functional forms for bonded and nonbonded interactions:

$$E = \sum_{i} k_r(r_i - r_0)^2 + \sum_{j} k_\theta(\theta_j - \theta_0)^2 + \sum_{k} V_n(1 + \cos(n\phi_k - \gamma)) + \sum_{l,m} \left( \frac{A_{lm}}{r_{lm}^{12}} - \frac{B_{lm}}{r_{lm}^6} \right)$$

These forms cannot describe bond breaking, electronic rearrangement, or complex many-body effects. They require extensive parameterization for each atom type and chemical environment.

### Neural Network Potentials (NNPs)

An NNP is a learnable function that maps local atomic environments to energies:

$$E_{\text{total}} = \sum_i E_i^{\text{atomic}}(\text{environment}_i)$$

where $E_i^{\text{atomic}}$ is a neural network trained on reference data (typically from quantum chemistry).

**Forces** are computed via automatic differentiation:

$$\mathbf{F}_i = -\nabla_{\mathbf{r}_i} E_{\text{total}}$$

This is one of the main advantages: gradients of the energy come "for free" from autograd, enabling molecular dynamics without explicit force parameterization.

### Descriptors and Invariance

The input to an NNP must encode local atomic geometry in a way that is:
- **Translation-invariant**: shifts in global coordinates do not change the energy
- **Rotation-invariant**: the energy is independent of molecular orientation
- **Permutation-invariant**: equivalent atoms can be reordered without changing the result

#### Behler-Parrinello Symmetry Functions (2007)

Atom-centered radial and angular descriptors:

**Radial:** $G_i^{\text{rad}}(\eta, R_s) = \sum_j \exp(-\eta (r_{ij} - R_s)^2)$

**Angular:** $G_i^{\text{ang}}(\eta, \zeta, \lambda) = 2^{1-\zeta} \sum_{j,k} (1 + \lambda \cos(\theta_{ijk}))^\zeta \exp(-\eta(r_{ij}^2 + r_{ik}^2 + r_{jk}^2))$

These are smooth functions of distances and angles, automatically invariant by construction.

#### Equivariant Message-Passing GNNs

Modern approaches use graph neural networks (GNNs) that operate on the atomic graph and maintain equivariance to rotations and translations:
- **SchNet (2017)**: continuous-filter convolutions, scalar-only functions
- **NequIP (2022)**: equivariant tensor representations, learns higher-order features
- **MACE (2023)**: higher-order equivariant message passing, pre-trained models available

### Key NNP Models

- **ANI** (Behler-type for C, H, O, N): small organic molecules, fast inference
- **MACE-OFF23**: state-of-the-art for organic chemistry, pre-trained on large datasets
- **PhysNet**: direct 3D coordinates, handles long-range interactions
- **DimeNet**: directional message passing, improves 3-body effects

### Practical Use

NNP inference is integrated into standard simulation packages:
- **OpenMM-ML**: plugin for OpenMM with ANI and other models
- **ASE (Atomic Simulation Environment)**: Python interface, supports multiple NNP models
- **LAMMPS**: pair style `nn/pair` for NNP forces

Basic usage: once trained or downloaded, running MD with an NNP is as simple as specifying the potential in your simulation script.

### Limitations

- **Transferability**: NNPs perform well only on geometries similar to training data. Out-of-distribution configurations (e.g., dissociation) may give unreliable results.
- **Data requirements**: training requires thousands to millions of quantum chemistry single-point calculations.
- **Accuracy vs. cost trade-off**: more parameters increase accuracy but also computational cost and training time.

## 10. Neural Networks for Collective Variables

### The Collective Variable (CV) Problem

In molecular dynamics, a collective variable (CV) is a function of atomic coordinates that describes a process of interest: binding, folding, conformational change, etc. A good CV should:
- Distinguish relevant states
- Smooth over irrelevant degrees of freedom (solvent noise, atomic vibrations)
- Have clear kinetic interpretation

Historically, chemists selected CVs by hand: bond distances, angles, dihedral angles (phi/psi for proteins). However, for large systems with complex mechanisms, manual CV design becomes difficult or impossible.

### Supervised Approach: Deep-LDA

**Deep Linear Discriminant Analysis (Bonati et al., 2020)** uses labeled simulation data to learn a CV that maximizes state separation.

Given trajectories labeled as belonging to state A, B, C, etc., a neural network is trained to classify these states. The network output becomes the CV:
- **Advantages**: Fast, interpretable in terms of state populations
- **Disadvantages**: requires pre-classified states; may not capture kinetic barriers well

### Unsupervised Approach: Deep-TICA

**Deep Time-Independent Component Analysis (Bonati et al., 2021)** learns from unlabeled time-series data without prior knowledge of states.

It combines neural network feature learning with TICA, solving the generalized eigenvalue problem:
$$\mathbf{C}(\tau) \mathbf{v} = \lambda \mathbf{C}(0) \mathbf{v}$$

where $\mathbf{C}(t)$ is the time-lagged covariance of learned features. The slowest-relaxing components become the output CVs.
- **Advantages**: fully unsupervised, captures kinetic timescales directly
- **Disadvantages**: requires tuning of lag time, larger training data, less interpretable

### Integration with PLUMED

The **mlcolvar** project (Bonati et al., 2023, *J. Chem. Phys.*) provides PyTorch models as PLUMED actions. The CV function is embedded in the simulation loop:

```
PYTORCH_MODEL FILE=model.pt
CVS: ...
```

PLUMED then:
1. Reads atomic coordinates from MD
2. Passes them to the PyTorch model
3. Computes the CV value and its gradient (via autograd)
4. Uses the CV to apply biasing forces (umbrella sampling, metadynamics, etc.)

### Workflow

1. Collect simulation data (unlabeled trajectories for Deep-TICA, or labeled for Deep-LDA)
2. Train a neural network in Python (as in this notebook)
3. Export the model to TorchScript or ONNX
4. Use in PLUMED for enhanced sampling
5. Extract thermodynamics and kinetics from the resulting biased trajectories

See the mlcolvar notebook for end-to-end implementation.

## 11. Further Reading

| Reference | Key Content |
| --- | --- |
| Goodfellow, Bengio, Courville (2016) *Deep Learning* | Foundational theory; chapters on backprop, optimization, architectures |
| [PyTorch Documentation](https://pytorch.org/docs) | nn modules, autograd, distributed training |
| Behler & Parrinello (2007) *PRL* 98, 146401 | Symmetry functions, seminal NNP work |
| Schütt et al. (2017) *NeurIPS* | SchNet: continuous filters for 3D molecules |
| Batatia et al. (2022) *NeurIPS* | MACE: higher-order equivariant message passing |
| Bonati et al. (2020) *Nat. Commun.* 11, 5633 | Deep-LDA for collective variables |
| Bonati et al. (2021) *J. Chem. Phys.* 154, 244119 | Deep-TICA for slow dynamics |
| Bonati et al. (2023) *J. Chem. Phys.* 159, 014801 | mlcolvar: PyTorch models in PLUMED |
| Tribello et al. (2014) *Comput. Phys. Commun.* 185, 604 | PLUMED 2: enhanced sampling framework |